# Paso 2: Exploración de Datos (EDA)

Este cuaderno realiza un análisis exploratorio completo del dataset sintético generado en el paso anterior.  
El objetivo es conocer la distribución de las variables, detectar patrones y generar figuras listas para el TFM.

**Autor:** Roberto (Fisioterapeuta — TFM Master Universitario)  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import sys
import os

# Añadir la raíz del proyecto al path para poder importar desde src/
proyecto_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if proyecto_raiz not in sys.path:
    sys.path.insert(0, proyecto_raiz)

print(f"Directorio raíz del proyecto: {proyecto_raiz}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from src.variables import (
    VARIABLES,
    COLUMNAS_FUERZA,
    COLUMNAS_MOVILIDAD,
    COLUMNAS_CONTROL,
    COLUMNAS_CONTEXTO,
)

# Estilo global de gráficos
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["font.family"] = "DejaVu Sans"

# Rutas
DATOS_DIR = Path(proyecto_raiz) / "datos" / "sinteticos"
FIGURAS_DIR = Path(proyecto_raiz) / "figuras"
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas correctamente.")
print(f"Carpeta de figuras: {FIGURAS_DIR}")

---
## 1. Carga de datos

In [ ]:
ruta_csv = DATOS_DIR / "dataset_sintetico.csv"
df = pd.read_csv(ruta_csv)

print(f"Dataset cargado correctamente.")
print(f"  Filas (deportistas): {df.shape[0]}")
print(f"  Columnas (variables + objetivo): {df.shape[1]}")
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# Verificar que la variable objetivo esté presente
assert "riesgo_lesion" in df.columns, "ERROR: no se encontró la columna 'riesgo_lesion'."

# Orden de categorías para gráficos
ORDEN_RIESGO = ["bajo", "medio", "alto"]
COLORES_RIESGO = {"bajo": "#2ecc71", "medio": "#f39c12", "alto": "#e74c3c"}

df["riesgo_lesion"] = pd.Categorical(df["riesgo_lesion"], categories=ORDEN_RIESGO, ordered=True)

print("Distribución de la variable objetivo:")
print(df["riesgo_lesion"].value_counts().sort_index())

---
## 2. Estadísticas descriptivas por bloque

Se muestran las estadísticas básicas agrupadas por los cuatro bloques clínicos:  
**Fuerza**, **Movilidad**, **Control y Equilibrio** y **Contexto**.

In [ ]:
# Traducciones de las estadísticas de describe() al español
TRADUCCIONES_DESCRIBE = {
    "count": "N válidos",
    "mean": "Media",
    "std": "Desv. típica",
    "min": "Mínimo",
    "25%": "Percentil 25",
    "50%": "Mediana",
    "75%": "Percentil 75",
    "max": "Máximo",
}

def describir_bloque(df, columnas, nombre_bloque):
    """Muestra describe() de un bloque con cabeceras en español."""
    cols_presentes = [c for c in columnas if c in df.columns]
    if not cols_presentes:
        print(f"[{nombre_bloque}] No se encontraron columnas en el dataset.")
        return None
    resumen = df[cols_presentes].describe().round(2)
    resumen.index = [TRADUCCIONES_DESCRIBE.get(i, i) for i in resumen.index]
    # Renombrar columnas usando nombre_display del diccionario VARIABLES
    nuevos_nombres = {
        col: VARIABLES[col]["nombre_display"] if col in VARIABLES else col
        for col in cols_presentes
    }
    resumen.rename(columns=nuevos_nombres, inplace=True)
    return resumen

In [ ]:
print("=" * 60)
print("BLOQUE 1: FUERZA")
print("=" * 60)
describir_bloque(df, COLUMNAS_FUERZA, "Fuerza")

In [ ]:
print("=" * 60)
print("BLOQUE 2: MOVILIDAD")
print("=" * 60)
describir_bloque(df, COLUMNAS_MOVILIDAD, "Movilidad")

In [ ]:
print("=" * 60)
print("BLOQUE 3: CONTROL Y EQUILIBRIO")
print("=" * 60)
describir_bloque(df, COLUMNAS_CONTROL, "Control")

In [ ]:
print("=" * 60)
print("BLOQUE 4: CONTEXTO")
print("=" * 60)
describir_bloque(df, COLUMNAS_CONTEXTO, "Contexto")

---
## 3. Histogramas por variable coloreados por nivel de riesgo

Se visualizan las distribuciones de las **8 variables clave** del protocolo.  
Cada color representa un nivel de riesgo: verde = bajo, naranja = medio, rojo = alto.

In [ ]:
# Variables clave seleccionadas y sus etiquetas en español
VARIABLES_CLAVE = [
    ("cuadriceps_der",          "Cuádriceps der. (N)"),
    ("cuadriceps_izq",          "Cuádriceps izq. (N)"),
    ("isquiotibiales_der",      "Isquiotibiales der. (N)"),
    ("dorsiflexion_tobillo_der", "Dorsiflexión tobillo der. (°)"),
    ("ybalance_anterior_der",   "Y-Balance anterior der. (cm)"),
    ("single_leg_hop_der",      "Single-leg hop der. (cm)"),
    ("edad",                    "Edad (años)"),
    ("dolor_percibido_nrs",     "Dolor percibido NRS (0–10)"),
    ("indice_estres_descanso",  "Índice de estrés y descanso"),
]

# Filtrar las que realmente están en el dataset
VARIABLES_CLAVE = [(col, etq) for col, etq in VARIABLES_CLAVE if col in df.columns]
print(f"Variables a graficar: {len(VARIABLES_CLAVE)}")
for col, etq in VARIABLES_CLAVE:
    print(f"  {col} -> {etq}")

In [ ]:
n_vars = len(VARIABLES_CLAVE)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 4))
axes = axes.flatten()

for idx, (col, etiqueta) in enumerate(VARIABLES_CLAVE):
    ax = axes[idx]
    for nivel in ORDEN_RIESGO:
        datos_nivel = df.loc[df["riesgo_lesion"] == nivel, col].dropna()
        ax.hist(
            datos_nivel,
            bins=20,
            alpha=0.55,
            color=COLORES_RIESGO[nivel],
            label=f"Riesgo {nivel}",
            edgecolor="white",
            linewidth=0.4,
        )
    ax.set_title(etiqueta, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Valor", fontsize=9)
    ax.set_ylabel("Frecuencia", fontsize=9)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.tick_params(labelsize=8)

# Ocultar ejes sobrantes
for idx in range(n_vars, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    "Distribución de variables clave por nivel de riesgo de lesión",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()

ruta_fig = FIGURAS_DIR / "histogramas_riesgo.png"
fig.savefig(ruta_fig, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig}")
plt.show()

---
## 4. Mapa de calor de correlaciones

Se representan las correlaciones de Pearson entre todas las variables numéricas continuas del dataset.  
Los valores cercanos a **+1** (azul oscuro) indican correlación positiva fuerte;  
los cercanos a **-1** (rojo oscuro) indican correlación negativa fuerte.

In [ ]:
# Seleccionar solo columnas numéricas (excluir objetivo categórico y ordinal de baja cardinalidad)
cols_excluir = ["riesgo_lesion", "genero", "single_leg_squat_valgo_der", "single_leg_squat_valgo_izq",
                "historial_lesional", "perfil_exigencia_deportiva", "dolor_percibido_nrs"]
cols_numericas = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in cols_excluir
]

print(f"Variables incluidas en el mapa de calor: {len(cols_numericas)}")

matriz_corr = df[cols_numericas].corr(method="pearson")

# Etiquetas cortas para el heatmap
etiquetas_cortas = []
for col in cols_numericas:
    if col in VARIABLES:
        nombre = VARIABLES[col]["nombre_display"]
        # Truncar para que quepan en el eje
        if len(nombre) > 28:
            nombre = nombre[:26] + "."
        etiquetas_cortas.append(nombre)
    else:
        etiquetas_cortas.append(col)

fig, ax = plt.subplots(figsize=(16, 13))

mascara_superior = np.triu(np.ones_like(matriz_corr, dtype=bool), k=1)

sns.heatmap(
    matriz_corr,
    mask=mascara_superior,
    annot=False,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.3,
    linecolor="#e0e0e0",
    square=True,
    cbar_kws={"label": "Coeficiente de correlación de Pearson", "shrink": 0.7},
    xticklabels=etiquetas_cortas,
    yticklabels=etiquetas_cortas,
    ax=ax,
)

ax.set_title(
    "Mapa de calor de correlaciones entre variables continuas",
    fontsize=14, fontweight="bold", pad=16
)
ax.tick_params(axis="x", labelsize=7, rotation=45)
ax.tick_params(axis="y", labelsize=7, rotation=0)
plt.tight_layout()

ruta_fig = FIGURAS_DIR / "correlaciones.png"
fig.savefig(ruta_fig, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig}")
plt.show()

---
## 5. Comparación bilateral (boxplots)

Se comparan los valores del lado **derecho** (azul) frente al **izquierdo** (naranja)  
para las variables bilaterales más relevantes en la clínica fisioterapéutica.  
Las asimetrías entre lados son un indicador clave de riesgo de lesión.

In [ ]:
# Variables bilaterales a comparar
PARES_BILATERALES = [
    ("cuadriceps",          "Cuádriceps isométrico (N)"),
    ("isquiotibiales",      "Isquiotibiales isométrico (N)"),
    ("gluteo_medio",        "Glúteo medio (N)"),
    ("dorsiflexion_tobillo","Dorsiflexión tobillo (°)"),
    ("ybalance_anterior",  "Y-Balance anterior (cm)"),
]

# Filtrar pares que realmente existen en el dataset
PARES_BILATERALES = [
    (base, etq) for base, etq in PARES_BILATERALES
    if f"{base}_der" in df.columns and f"{base}_izq" in df.columns
]
print(f"Pares bilaterales a representar: {len(PARES_BILATERALES)}")

In [ ]:
n_pares = len(PARES_BILATERALES)
fig, axes = plt.subplots(1, n_pares, figsize=(n_pares * 3, 6), sharey=False)

if n_pares == 1:
    axes = [axes]

COLOR_DER = "#2980b9"   # azul — derecha
COLOR_IZQ = "#e67e22"   # naranja — izquierda

for ax, (base, etiqueta) in zip(axes, PARES_BILATERALES):
    col_der = f"{base}_der"
    col_izq = f"{base}_izq"

    # Construir dataframe largo para seaborn
    datos_der = df[[col_der]].rename(columns={col_der: "valor"})
    datos_der["Lado"] = "Derecha"
    datos_izq = df[[col_izq]].rename(columns={col_izq: "valor"})
    datos_izq["Lado"] = "Izquierda"
    datos_largo = pd.concat([datos_der, datos_izq], ignore_index=True)

    sns.boxplot(
        data=datos_largo,
        x="Lado", y="valor",
        palette={"Derecha": COLOR_DER, "Izquierda": COLOR_IZQ},
        order=["Derecha", "Izquierda"],
        width=0.5,
        linewidth=1.2,
        flierprops=dict(marker="o", markersize=3, alpha=0.4),
        ax=ax,
    )

    # Calcular y mostrar medias
    media_der = df[col_der].mean()
    media_izq = df[col_izq].mean()
    ax.axhline(media_der, color=COLOR_DER, linestyle="--", linewidth=0.9, alpha=0.7)
    ax.axhline(media_izq, color=COLOR_IZQ, linestyle="--", linewidth=0.9, alpha=0.7)

    ax.set_title(etiqueta, fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("")
    ax.set_ylabel("Valor", fontsize=9)
    ax.tick_params(labelsize=9)

# Leyenda global
parche_der = mpatches.Patch(color=COLOR_DER, label="Derecha")
parche_izq = mpatches.Patch(color=COLOR_IZQ, label="Izquierda")
fig.legend(
    handles=[parche_der, parche_izq],
    loc="upper center", ncol=2,
    fontsize=10, framealpha=0.8,
    title="Lado evaluado", title_fontsize=10,
    bbox_to_anchor=(0.5, 1.04),
)

fig.suptitle(
    "Comparación bilateral de variables de fuerza, movilidad y equilibrio",
    fontsize=13, fontweight="bold", y=1.08
)
plt.tight_layout()

ruta_fig = FIGURAS_DIR / "comparacion_bilateral.png"
fig.savefig(ruta_fig, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig}")
plt.show()

---
## 6. Distribución de la variable objetivo

Se muestra cuántos deportistas hay en cada categoría de riesgo  
(**bajo**, **medio**, **alto**) tanto en número absoluto como en porcentaje.

In [ ]:
conteo_riesgo = df["riesgo_lesion"].value_counts().reindex(ORDEN_RIESGO)
porcentajes = conteo_riesgo / conteo_riesgo.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Gráfico de barras (izquierda) ---
ax_bar = axes[0]
barras = ax_bar.bar(
    ORDEN_RIESGO,
    conteo_riesgo.values,
    color=[COLORES_RIESGO[n] for n in ORDEN_RIESGO],
    edgecolor="white",
    linewidth=1.5,
    width=0.5,
)
for barra, n, pct in zip(barras, conteo_riesgo.values, porcentajes.values):
    ax_bar.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + max(conteo_riesgo.values) * 0.01,
        f"{n}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )
ax_bar.set_title("Distribución de niveles de riesgo", fontsize=12, fontweight="bold")
ax_bar.set_xlabel("Nivel de riesgo de lesión", fontsize=10)
ax_bar.set_ylabel("Número de deportistas", fontsize=10)
ax_bar.set_xticklabels(["Bajo", "Medio", "Alto"], fontsize=11)
ax_bar.set_ylim(0, max(conteo_riesgo.values) * 1.15)
ax_bar.tick_params(labelsize=10)

# --- Diagrama de sectores (derecha) ---
ax_pie = axes[1]
wedges, texts, autotexts = ax_pie.pie(
    conteo_riesgo.values,
    labels=["Bajo", "Medio", "Alto"],
    autopct="%1.1f%%",
    colors=[COLORES_RIESGO[n] for n in ORDEN_RIESGO],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    pctdistance=0.78,
)
for text in texts:
    text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight("bold")
    autotext.set_color("white")
ax_pie.set_title("Proporción de cada nivel de riesgo", fontsize=12, fontweight="bold")

fig.suptitle(
    f"Variable objetivo: riesgo_lesion (N total = {len(df)} deportistas)",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()

ruta_fig = FIGURAS_DIR / "distribucion_riesgo.png"
fig.savefig(ruta_fig, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig}")
plt.show()

---
## 7. Resumen

El análisis exploratorio ha generado las siguientes figuras:

| Archivo | Contenido |
|---|---|
| `histogramas_riesgo.png` | Distribución de 9 variables clave coloreadas por nivel de riesgo |
| `correlaciones.png` | Mapa de calor de correlaciones de Pearson entre variables continuas |
| `comparacion_bilateral.png` | Boxplots comparativos derecha vs. izquierda para variables bilaterales |
| `distribucion_riesgo.png` | Diagrama de barras y sectores de la variable objetivo |

**Estas figuras están listas para incluir en tu TFM. Las encontrarás en la carpeta `figuras/`**

---

### Observaciones clave a destacar en el TFM

- **Histogramas**: observa si las distribuciones de los grupos de riesgo se solapan o se separan claramente. Una buena separación indica que la variable tiene poder discriminante.
- **Correlaciones**: las variables del mismo bloque (p. ej., cuádriceps derecho e izquierdo) suelen estar muy correlacionadas entre sí. Variables de bloques distintos con alta correlación pueden indicar relaciones clínicas interesantes.
- **Comparación bilateral**: asimetrías entre lados (der vs. izq) por encima del 10–15 % son clínicamente relevantes y se asocian a mayor riesgo de lesión según la literatura.
- **Variable objetivo**: si la distribución está muy desequilibrada (p. ej., 70 % bajo riesgo), habrá que aplicar técnicas de balanceo de clases (SMOTE, pesos de clase) en el paso de modelado.

El siguiente paso es el **preprocesado y preparación de datos** (`03_preprocesado.ipynb`).